## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Chatbot vs human agent tickets: median test

**Median Test** · a worked example from the StatNex teaching guide

**The story.** An online shop answers support tickets two ways: a chatbot (18 tickets) and a human agent (9 tickets). Most chatbot tickets are solved in minutes, but the ones the bot cannot handle sit for close to an hour.

**The question.** Does a typical chatbot ticket get solved faster than a typical human-agent ticket?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/median_test.csv` (18 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Chatbot,Human agent
5.0,14.0
30.0,12.0
4.0,16.0
7.0,13.0
6.0,19.0
42.0,11.0
3.0,15.0
5.0,17.0
9.0,13.0
6.0,
55.0,
4.0,
8.0,
7.0,
6.0,
5.0,
9.0,
70.0,
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
# One column per sample; text and blank cells are skipped, so the samples may differ
# in size.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
samples1 = ["Chatbot"]
samples2 = ["Human agent"]
# Each column once, in the order the pairs below use them.
columns = list(dict.fromkeys(name for a in samples1 for b in samples2 for name in (a, b)))
values = {name: pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)
          for name in columns}

## Descriptive statistics

In [ ]:
descriptive = pd.DataFrame({"n": [len(v) for v in values.values()],
                            "median": [np.median(v) for v in values.values()]},
                           index=list(values))
print(descriptive)

## Median test

In [ ]:
# Each observation is reduced to one bit -- which side of the pooled median it lies
# on -- and the 2 x 2 table of counts is tested for independence. Every column of
# sample 1 is tested against every column of sample 2.
tests, tables = [], []
for a in samples1:
    for b in samples2:
        x, y = values[a], values[b]
        # scipy.stats.median_test is Mood's median test: it finds the pooled median,
        # counts each sample above and not above it, and runs the chi-square test of
        # independence on that 2 x 2 table. Its table has a row "above" and a row "not
        # above"; it is turned round here so that each sample is a row.
        # A value equal to the median counts as "at or below".
        plain = stats.median_test(x, y, ties="below", correction=False)
        sides = ["at or below the median", "above the median"]
        median, chi2, p, df = plain.median, plain.statistic, plain.pvalue, 1
        counts = plain.table[::-1].T
        # Yates' correction takes 0.5 off each |observed - expected| before squaring. It
        # makes the chi-square smaller and the P value larger (more cautious).
        chi2_yates, p_yates, _, _ = stats.chi2_contingency(counts, correction=True)
        # Every expected count should be 5 or more for the chi-square P value to be
        # trusted. When some are smaller, read Fisher's exact test, which counts every
        # table with the same totals instead of using the chi-square curve.
        fisher_two = stats.fisher_exact(counts).pvalue
        # The one-sided Fisher test looks in the direction the table leans, so it gets no
        # stars: starring it would halve the two-sided P after looking.
        leans = "greater" if counts[0, 0] * counts[1, 1] >= counts[0, 1] * counts[1, 0] else "less"
        fisher_one = stats.fisher_exact(counts, alternative=leans).pvalue
        test = pd.DataFrame(
            [["chi-square", chi2, df, p, stars(p)],
             ["Yates' correction", chi2_yates, df, p_yates, stars(p_yates)],
             ["Fisher exact, two-sided", np.nan, np.nan, fisher_two, stars(fisher_two)],
             ["Fisher exact, one-sided", np.nan, np.nan, fisher_one, ""]],
            columns=["method", "chi-square", "df", "P", ""])
        table = pd.DataFrame(counts, index=[a, b], columns=sides)
        table["total"] = table.sum(axis=1)
        table.loc["total"] = table.sum()
        tests.append(test)
        tables.append(table)
        print(f"\n{a} vs {b}: pooled median = {median:g}")
        print(test.to_string(index=False))
        print(table)

## What the result says

The median is the middle value: half of the tickets are solved faster and half slower. The median time is 6.5 minutes for the chatbot and 14 minutes for the human agent.

The median test puts all 27 tickets together, finds their pooled median of 9 minutes, and counts each group's tickets on each side of it. Of the 18 chatbot tickets, 14 are at or below 9 minutes. Of the 9 human agent tickets, none is. The chi-square statistic for this table is 14.54, and Fisher's exact test, which suits small counts, gives a P value of 0.00015. The P value is the chance of a split this uneven if both channels had the same median.

So a typical chatbot ticket is solved faster. But the test only counts sides: it says nothing about how long the 4 slow chatbot tickets took.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.